# SMART · From clinical narrative to aging-state analysis

This walkthrough combines three constructed clinical inputs, the exact task prompts, linked-field counterfactual changes and actual study-level summary results. It runs offline. Load saved predictions to calculate aging-state summaries. No numerical ages are assigned to the example participants in the offline demonstration.


In [ ]:
from pathlib import Path
import sys, json, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "smart/templates/age_estimation.txt").exists())
sys.path.insert(0, str(ROOT))
from smart.scn import render_scn
from smart.prompts import build_age_prompt, build_recommendation_prompt
from smart.metrics import summarize_predictions
from smart.counterfactual import perturb_measurements
domains = json.loads((ROOT / "examples/age_domains.json").read_text(encoding="utf-8-sig"))


## 1. Inspect complementary clinical inputs

The first input combines respiratory abnormalities with clinical context. The second has low AHI but different continuity and architecture measurements. The third preserves home-monitoring REI and unknown sleep staging. Their purpose is to demonstrate input coverage, not to establish an organ-specific response in advance.


In [ ]:
cases = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "examples").glob("0*.json"))}
for name, record in cases.items():
    display(Markdown("### " + name.replace("_", " ")))
    print(render_scn(record))


## 2. Prepare the age-estimation prompt

The requested domains can be changed explicitly. The example below uses the complete output list. Clinical information is preserved in the narrative. Diagnosis-discrimination studies require their specified history masking before constructing an input.


In [ ]:
selected_case = "01_respiratory_context"
scn = render_scn(cases[selected_case])
prompt = build_age_prompt(domains, scn)
print(prompt)


## 3. Trace counterfactual input changes

The offline scenarios modify AHI in the respiratory example and TST in the continuity example. Dependent indices and sleep efficiency are recalculated. In the constructed continuity example, the 420-min recording spans lights off to lights on. Increasing TST from 300 to 360 min reduces WASO from 90 to 30 min while sleep onset latency remains 30 min and terminal wake time remains zero. These timing constraints are applied by the example wrapper. Baseline and modified narratives must both be estimated with the same greedy configuration when generating a response table.

In [ ]:
spec = importlib.util.spec_from_file_location("smart_demo", ROOT / "examples/run_demo.py")
demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo)
for name, target, value in [("01_respiratory_context", "AHI", 5), ("02_sleep_continuity", "TST", 360)]:
    changed, audit = demo.scenario(cases[name], target, value)
    display(Markdown("### " + name))
    display(pd.DataFrame(audit))
demo.run()


## 4. Understand the three calculations

This small arithmetic exercise is separate from the clinical examples. Its explicitly constructed numbers test the distinction between variation across organs and across decodings. They are not LLM predictions or study observations. The overall sleep estimate is excluded from organ asynchrony and organ instability.


In [ ]:
arithmetic_domains = ["overall sleep biological age", "cardiovascular age", "respiratory age", "metabolic age"]
centres = np.array([60., 63., 57., 60.])
offsets = np.arange(-4.5, 5.0) / 10
constructed = (centres[None, :] + offsets[:, None] * np.array([1., 2., 1., 0.5]))[None, :, :]
summary = summarize_predictions(constructed, [60.], arithmetic_domains)
display(pd.DataFrame({"domain": arithmetic_domains, "mean_age": summary["mean_age"][0],
                      "age_gap": summary["age_gap"][0], "decoding_SD": summary["decoding_sd"][0],
                      "MCSE": summary["mcse"][0]}))
print("Constructed calculation only:")
print("Asynchrony:", summary["asynchrony"][0], "years")
print("Instability:", summary["instability"][0], "years")


## 5. Examine actual study summaries

Unlike the illustrative inputs above, these two tables contain study results. Every organ/outcome and strategy in the supplied source tables is shown. The middle group is the mean of ten separately evaluated performances, whereas the ensemble group evaluates mean predictions. CIs were obtained from 2,000 paired participant bootstrap resamples. Study data access is required to reproduce the original participant-level analyses.


In [ ]:
for filename in ["fig4h_decoding_auc.csv", "ed9c_decoding_cindex.csv"]:
    table = pd.read_csv(ROOT / "analyses/data" / filename)
    display(Markdown("### " + filename))
    display(table)


## 6. Connect real predictions and recommendation generation

`examples/run_demo.py --predictions PATH --case CASE_NAME` loads those responses and computes aging-state summaries. The optional cell below stays inactive until a genuine saved response file is supplied.

For recommendations, generate a baseline and counterfactual age table with the same greedy settings. Do not substitute the arithmetic exercise above for model estimates. Scenario labels must name the modified input.

The recommendation template requires saved age estimates for one individual’s baseline and at least nine distinct valid counterfactual scenarios. The two walkthrough scenarios concern different constructed individuals and must not be combined into one recommendation table.

In [ ]:
PREDICTIONS = None  # Path to saved_predictions.json for selected_case, if available
if PREDICTIONS is not None:
    demo.run(predictions=PREDICTIONS, case=selected_case)
else:
    print("Optional real-prediction branch not executed.")

COUNTERFACTUAL_TABLE = None  # Path to a genuine labelled baseline/counterfactual age table
if COUNTERFACTUAL_TABLE is not None:
    table_text = Path(COUNTERFACTUAL_TABLE).read_text(encoding="utf-8")
    recommendation_prompt = build_recommendation_prompt(scn, table_text)
    print(recommendation_prompt)
else:
    print("Recommendation template:", ROOT / "smart/templates/recommendation.txt")
